In [1]:
# Campaign 3: Emotions
emotions_labels = {
    "E01": "focus",
    "E02": "distraction",
    "E03": "stress",
    "E04": "relaxation",
    "E05": "depression",
    "E06": "excitement"
}
decimals = 1

In [2]:
#Radar
import pickle
import numpy as np
import os
import pandas as pd

class RadarSensor:
    """A container for a single radar unit's data and configuration."""

    def __init__(self, translation, transform_func):
        self.translation = translation
        self.transform_func = transform_func  # Function to handle coordinate mapping

    def _load_data(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            # Filter empty or invalid frames (logic from ground script)
            cleaned = [arr for arr in data if isinstance(arr, np.ndarray) and arr.size > 0]
            # Fallback if cleaning removed everything or list was raw (logic from ceiling script)
            return np.concatenate(cleaned, axis=0) if cleaned else data
        
        
    def load_and_transform_data(self, path):
        """Loads data from a file and applies the transformation."""
        frames = self._load_data(path)
        return self.transform_func(frames, self.translation)
        
    
        
# --- Specific Math Logic for Ceiling ---
def ceiling_transform(points, translation):
    time_stamp=points[:, 0:1]  # First column is timestamp
    # Logic from original script: z = -points[:,1], y = points[:,2], x = points[:,3]
    z = -points[:, 1]
    y = points[:, 2]
    x = points[:, 3]
    translated = np.stack([x, y, z], axis=1) + translation
    return np.hstack([time_stamp, translated])

# --- Specific Math Logic for Ground ---
def ground_transform_factory(deg):
    """Returns a transformation function locked to a specific angle."""
    theta_rad = np.deg2rad(deg)
    # Pre-calculate R matrix
    R = np.array([
        [np.cos(theta_rad), -np.sin(theta_rad), 0],
        [np.sin(theta_rad), np.cos(theta_rad), 0],
        [0, 0, 1]
    ])

    def transform(points, translation):
        time_stamp=points[:, 0:1]  # First column is timestamp
        # Apply rotation then translation
        # Note: points[:, 1] is X, points[:, 2] is Y, points[:, 3] is Z
        res = R @ [points[:, 1], points[:, 2], points[:, 3]]
        x, y, z = res[0], res[1], res[2]
        translated = np.stack([x, y, z], axis=1) + translation
        return np.hstack([time_stamp, translated])

    return transform

In [3]:
sensors_info=[]

#Ceiling
FILES = ['0.pkl', '1.pkl', '2.pkl', '3.pkl', '4.pkl']
TRANSLATIONS = [
    np.array([-2, 4, 5]), np.array([2, 4, 5]), np.array([0, 0, 5]),
    np.array([-2, -4, 5]), np.array([2, -4, 5])
]
sensors_info.extend(list(zip(FILES, TRANSLATIONS,[ceiling_transform for _ in FILES])))

#Ground
FILES = ['5.pkl', '6.pkl', '7.pkl', '8.pkl', '9.pkl', '10.pkl', '11.pkl', '12.pkl']
THETA_DEGS = [180, 135, 90, 45, 0, -45, -90, -135]
TRANSLATIONS = [
    np.array([1.5, 0, 1.3]), np.array([1.06, -1.06, 1.3]),
    np.array([0, -1.5, 1.3]), np.array([-1.06, -1.06, 1.3]),
    np.array([-1.5, 0, 1.3]), np.array([-1.06, 1.06, 1.3]),
    np.array([0, 1.5, 1.3]), np.array([1.06, 1.06, 1.3])
]
sensors_info.extend(list(zip(FILES, TRANSLATIONS,[ground_transform_factory(degree) for degree in THETA_DEGS])))

In [4]:
sensors = []
for file_name, translation, transform_func in sensors_info:
    s = RadarSensor(
        translation=translation,
        transform_func=transform_func
    )
    sensors.append(s)

In [5]:
columns = []
for i in range(13):
    columns.extend([f"x_{i}", f"y_{i}", f"z_{i}"])

In [6]:
import glob

frame_data={}
users = glob.glob(f"E:/CoDaS Project/Radar/C3/*")
for user in users:
    user_id = os.path.basename(user)
    frame_data[user_id] = {}
    for class_key in emotions_labels.keys():
        path = f"{user}/{class_key}/*/*.pkl"
        pkl_files = glob.glob(path)
        
        frame_data[user_id][class_key] = {}
        merged_df=None
        for file in pkl_files:
            sensor_id=os.path.basename(file).replace('.pkl', '')
            sensor_id=int(sensor_id)
            frames = sensors[sensor_id].load_and_transform_data(file)

            frames = pd.DataFrame(frames, columns=['timestamp', 'x', 'y', 'z'])

            frames = frames.sort_values(by='timestamp')  # Ensure data is sorted by timestamp

            frames['timestamp'] = frames['timestamp'].round(decimals)  # Round timestamps to the specified number of decimal places for consistency
            frames = frames.groupby("timestamp", as_index=True).mean()

            #frames = frames.set_index('timestamp')
            frames.to_csv(f"{user}/{class_key}/sensor_{sensor_id}_transformed.csv")  # Save transformed data as CSV
            frame_data[user_id][class_key][sensor_id] = frames

            frames.rename(columns={'x': f'x_{sensor_id}', 'y': f'y_{sensor_id}', 'z': f'z_{sensor_id}'}, inplace=True)
            if merged_df is None:
                merged_df = frames
            else:
                merged_df = merged_df.join(frames, how='outer')

        if merged_df is not None:
            sorted_merged_df = merged_df.sort_index()  # Ensure the final merged DataFrame is sorted by timestamp

            #Ensure all expected columns are present, filling missing ones with NaN
            for i in range(13):
                if f'x_{i}' not in sorted_merged_df.columns:
                    sorted_merged_df[f'x_{i}'] = np.nan
                    sorted_merged_df[f'y_{i}'] = np.nan
                    sorted_merged_df[f'z_{i}'] = np.nan

            sorted_merged_df = sorted_merged_df.loc[:,columns]

            sorted_merged_df = sorted_merged_df.reset_index()
            sorted_merged_df['timestamp'] = sorted_merged_df['timestamp'] - sorted_merged_df['timestamp'].min()  # Normalize timestamps to start at 0
            
            sorted_merged_df.to_csv(f"{user}/{class_key}/merged_sensors.csv", index=False)
            frame_data[user_id][class_key]['merged'] = sorted_merged_df

In [7]:
all_data = []
for user_id, emotions in frame_data.items():
    for emotion, data in emotions.items():
        if 'merged' in data:
            df = data['merged'].copy()
            
            df.insert(0, 'user_id', user_id)
            df.insert(1, 'emotion', emotion)
            all_data.append(df)
radar_df = pd.concat(all_data, ignore_index=False)
radar_df.to_csv("E:/CoDaS Project/Radar/C3/radar_merged_data.csv", index=False)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer
import glob

columns = ["timestamp"]
for i in range(13):
    columns.extend([f"x_{i}", f"y_{i}", f"z_{i}"])

    
spf = 0.1  # reciprocal of frames per second
users = glob.glob(f"E:/CoDaS Project/Radar/C3/*")
users = [p for p in glob.glob(r"E:/CoDaS Project/Radar/C3/*") ]
merged_interp_df=[]
for user in users:
    if not os.path.isdir(user):
        continue
    user_id = os.path.basename(user)
    for class_key in emotions_labels.keys():
        if not os.path.exists(f"{user}/{class_key}"):
            continue
        path = f"{user}/{class_key}/*/*.pkl"
        pkl_files = glob.glob(path)
        
        merged_df=None

        #Get the overall start and end timestamps across all sensors for this user and emotion class
        sync_start=0
        sync_end=np.inf
        sensor_interpolators={}
        for sensor_id in range(13):
            if os.path.exists(f"{user}/{class_key}/sensor_{sensor_id}_transformed.csv"):
                frames=pd.read_csv(f"{user}/{class_key}/sensor_{sensor_id}_transformed.csv")
                start, end = frames["timestamp"].min(), frames["timestamp"].max()

                # Avoid operations on empty dataframes, 10 samples is an arbitrary threshold
                if len(frames) < 10:
                    continue

                sync_start = max(sync_start, start)
                sync_end = min(sync_end, end)

                x_train = frames["timestamp"].values.reshape(-1, 1)
                y_train = frames[["x", "y", "z"]].values


                model = make_pipeline(SplineTransformer(n_knots=4, degree=3), Ridge(alpha=1e-3))
                model.fit(x_train, y_train)
                sensor_interpolators[sensor_id] = model


        # Interpolate at regular intervals and create recurrence plots
        interp_timestamps = np.arange(sync_start, sync_end, step=spf).reshape(-1, 1)
        interp_sensor_data = model.predict(interp_timestamps)
        interpolated_data = {'timestamp': interp_timestamps.flatten()}
        for sensor_id, model in sensor_interpolators.items():
            interp_values = model.predict(interp_timestamps)
            interpolated_data[f'x_{sensor_id}'] = interp_values[:, 0]
            interpolated_data[f'y_{sensor_id}'] = interp_values[:, 1]
            interpolated_data[f'z_{sensor_id}'] = interp_values[:, 2]

                
        interp_df = pd.DataFrame(interpolated_data)
        interp_df['timestamp'] = interp_df['timestamp'] - interp_df['timestamp'].min()  # Normalize timestamps to start at 0
        interp_df['timestamp'] = interp_df['timestamp'].round(2)  # Round timestamps to 3 decimal places for cleaner output

        interp_df = interp_df.reindex(columns=columns)
        interp_df.to_csv(f"{user}/{class_key}/interpolated_sensors.csv", index=False)

        interp_df.insert(0, 'user_id', user_id)
        interp_df.insert(1, 'emotion', class_key)
        merged_interp_df.append(interp_df)
radar_interp_df = pd.concat(merged_interp_df, ignore_index=False)
radar_interp_df.to_csv("E:/CoDaS Project/Radar/C3/radar_interpolated_merged_data.csv", index=False)

In [ ]:
radar_interp_df.head()